In [4]:
import json
import pickle
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

# --- 1. Load clean data and build prepared_text per row ---
with open("../data/processed/master_bank_knowledge.json", "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)
df["prepared_text"] = (
    "Category: " + df["sheet"].astype(str)
    + " | Topic: " + df["topic"].astype(str)
    + " | Question: " + df["question"].astype(str)
    + " | Answer: " + df["answer"].astype(str)
)

# --- 2. Recursive character chunking ---
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

text_mapping = {}
chunk_id = 0

for row_idx, row in df.iterrows():
    source_question = str(row["question"])
    source_sheet = str(row["sheet"])
    source_topic = str(row["topic"])
    text = str(row["prepared_text"])
    for chunk_text in splitter.split_text(text):
        text_mapping[chunk_id] = {
            "chunk_text": chunk_text,
            "question": source_question,
            "sheet": source_sheet,
            "topic": source_topic,
            "source_row_index": int(row_idx)
        }
        chunk_id += 1

chunk_texts = [text_mapping[i]["chunk_text"] for i in range(len(text_mapping))]

from tqdm.auto import tqdm

print("Loading embedding model...")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

print(f"Encoding {len(chunk_texts)} chunks...")
BATCH = 64
parts = []

for start in tqdm(range(0, len(chunk_texts), BATCH)):
    batch = chunk_texts[start:start+BATCH]
    part = embed_model.encode(batch, convert_to_numpy=True)
    parts.append(np.asarray(part, dtype=np.float32))

vectors = np.vstack(parts)

dimension = vectors.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(vectors)

out_dir = Path("../data/processed")
out_dir.mkdir(parents=True, exist_ok=True)

faiss.write_index(index, str(out_dir / "bank_faiss.index"))

with open("../data/processed/text_mapping.pkl", "wb") as f:
    pickle.dump(text_mapping, f)

print("Done!")


Loading embedding model (first run can take several minutes downloading weights)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |
------------------------+------------+-
embeddings.position_ids | UNEXPECTED |

Notes:
- UNEXPECTED: can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding 381 chunks...


Embedding batches:   0%|          | 0/6 [00:00<?, ?batch/s]

Output directory is: C:\\nust-bank-llm-assistant\\data\\processed
FAISS index shape: 381 vectors, dimension 384
Will write FAISS index to: ..\\data\\processed\\bank_faiss.index
Will pickle text_mapping (381 entries) to: ..\\data\\processed\\text_mapping.pkl


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

✅ Successfully saved both FAISS index and Pickle mapping!
Done: 381 chunks indexed -> ..\\data\\processed\\bank_faiss.index
